In [ ]:
import feedparser
import requests

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'Accept-Language': 'en-US,en;q=0.9'
}

response = requests.get('https://www.scmp.com/topics/hong-kong-property', headers=headers)
feed = feedparser.parse(response.content)

for entry in feed.entries[:5]:  # Show latest 5 transactions
    print(f"{entry.published}: {entry.title}\n{entry.link}\n")


In [19]:
import requests
api_url = "https://api.data.gov.hk/v1/historical-archive/list-files"
params = {
    'start': '20250501',
    'end': '20250525', # latest: yesterday
    'category': 'housing'
}
response = requests.get(api_url, params=params)
print(response.json())


{'file-count': 1122, 'files': [{'dataset-id': 'centaline-centanetod-ccipropertyinfo', 'dataset-name-en': 'Property information of the CCI constituent estates', 'dataset-name-tc': 'CCI成份屋苑的物業資料', 'dataset-name-sc': 'CCI成份屋苑的物业资料', 'resource-name-en': 'Property information of the CCI constituent estates', 'resource-name-tc': 'CCI成份屋苑的物業資料', 'resource-name-sc': 'CCI成份屋苑的物业资料', 'data_dictionary': 'http://hk.centanet.com/opendata/Data-Dictionary%20-%20CCI%20Estate.pdf', 'schema': '', 'provider-id': 'centaline', 'category-id': 'housing', 'format': 'csv', 'url': 'http://hk.centanet.com/opendata/CCI%20Estate%20for%20Opendata.csv', 'version-count': 1, 'total-size': 58363}, {'dataset-id': 'hk-lr-data1-landreg', 'dataset-name-en': 'Monthly statistics on instruments received for registration (April 1993-December 2000)', 'dataset-name-tc': '送交註冊文書的每月統計數字(1993年4月-2000年12月)', 'dataset-name-sc': '送交注册文书的每月统计数字(1993年4月-2000年12月)', 'resource-name-en': 'Statistics for the month of April 1993', 'resource-

In [10]:
class PropertySpider(scrapy.Spider):
    name = 'hk_properties'
    start_urls = ['https://www.scmp.com/property']
    
    def parse(self, response):
        yield {
            'title': response.css('h1.article__title::text').get(),
            'transaction_value': response.xpath('//div[@class="price"]/text()').get()
        }


NameError: name 'scrapy' is not defined

In [12]:
from bs4 import BeautifulSoup
import requests

html = requests.get('https://www.landsd.gov.hk').text
soup = BeautifulSoup(html, 'lxml')
transactions = soup.select('div.transaction-record')
transactions

[]

In [15]:
# Solution: Implement proper headers
import feedparser

feed = feedparser.parse(
    'https://www.scmp.com/property',
    request_headers={
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
        'Accept-Language': 'en-US,en;q=0.9',
        'Referer': 'https://www.scmp.com/'
    }
)
feed

{'bozo': 1,
 'entries': [],
 'feed': {'html': {'data-commit-sha': 'bffaffbc673c09ffd227a4bc2f5445d58feaefbe',
   'data-is-bot': 'false',
   'lang': 'en',
   'data-qa': 'Document-Html'},
  'head': {'data-qa': 'Document-Head'},
  'meta': {'name': 'baggage',
   'content': 'sentry-environment=production,sentry-release=bffaffbc673c09ffd227a4bc2f5445d58feaefbe,sentry-public_key=13c321220cc471acba5c6d75db746859,sentry-trace_id=09cad4666c76e12b723b862f29aa5e6f'},
  'links': [{'rel': 'canonical',
    'href': 'https://www.scmp.com/property',
    'data-next-head': '',
    'type': 'text/html'},
   {'href': 'https://www.scmp.com/static/manifest.json',
    'rel': 'manifest',
    'data-next-head': '',
    'type': 'text/html'},
   {'href': 'https://assets-v2.i-scmp.com/production/favicon.ico',
    'rel': 'shortcut icon',
    'data-next-head': '',
    'type': 'text/html'},
   {'href': 'https://assets-v2.i-scmp.com/production/icons/scmp-icon-192x192.png',
    'rel': 'icon',
    'sizes': '192x192',
    '

In [28]:
# Modified SCMP parser with proper SSL handling
import feedparser
import requests
from datetime import datetime

def parse_scmp_property_feed():
    url = 'https://www.scmp.com/topics/hong-kong-property'  # Correct RSS endpoint
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
        'Accept': 'application/rss+xml',
        'Referer': 'https://www.scmp.com/property'
    }

    try:
        # Handle SSL at request level
        response = requests.get(url, 
                              headers=headers, 
                              timeout=10,
                              verify=True)  # Enable SSL verification
        
        # Check feed parsing errors
        feed = feedparser.parse(response.content)
        if feed.bozo:
            print(f"Feed parsing error: {feed.bozo_exception}")

        # Extract property news
        for entry in feed.entries:
            pub_date = datetime(*entry.published_parsed[:6]).strftime('%Y-%m-%d %H:%M')
            print(f"【SCMP Property】{pub_date}")
            print(f"Title: {entry.title}")
            print(f"Link: {entry.link}")
            print(f"Summary: {entry.description[:150]}...\n")

    except Exception as e:
        print(f"SCMP Error: {str(e)}")


In [29]:
import requests
from bs4 import BeautifulSoup

# Modified HKEJ parser with proper feed handling
def parse_hkej_property_news():
    url = 'https://www2.hkej.com/property/rss'  # Verified RSS endpoint
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
        'Accept-Language': 'zh-HK,zh;q=0.9'
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.encoding = 'utf-8'  # Force correct encoding
        
        feed = feedparser.parse(response.content)
        
        if not feed.entries:
            print("HKEJ Feed appears empty. Checking alternatives...")
            # Fallback to HTML parsing if RSS fails
            soup = BeautifulSoup(response.text, 'lxml')
            items = soup.find_all('item')
            for item in items[:5]:  # Show first 5 entries
                title = item.find('title').text.strip()
                link = item.find('link').text.strip()
                print(f"【HKEJ Property】{title}\n{link}\n")
            return

        for entry in feed.entries:
            if '地產' in entry.get('category', '') or 'property' in entry.title.lower():
                pub_date = datetime(*entry.published_parsed[:6]).strftime('%Y-%m-%d %H:%M')
                print(f"【HKEJ Property】{pub_date}")
                print(f"Title: {entry.title}")
                print(f"Link: {entry.link}")
                print(f"Summary: {entry.description[:150]}...\n")

    except Exception as e:
        print(f"HKEJ Error: {str(e)}")


parse_hkej_property_news()


HKEJ Feed appears empty. Checking alternatives...
